In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error 
from sklearn.linear_model import ElasticNetCV , LassoCV , RidgeCV
from sklearn.linear_model import ElasticNet , Lasso , Ridge
import plotly.graph_objects as go



In [ ]:
n=1000
p=5000
eta_vec = np.random.normal(0,1, n)
beta=np.zeros(p)
beta[:15]=1


In [ ]:
uni_beta , counts=np.unique(beta , return_counts=True)
for value, count in zip(uni_beta , counts):
    print(f"Value : {value} , Count : {count}")

In [ ]:

X= np.random.randn(n,p)

Y=np.matmul(X, beta)+eta_vec 

In [ ]:
x_train, x_test , y_train , y_test = train_test_split(X , Y , test_size=0.34 , random_state=42)

In [ ]:
print("Training_sample shape:", x_train.shape)
print("test_sample shape:", x_test.shape)
print("Training_labels shape:", y_train.shape)
print("test_labels shape:", y_test.shape)

In [ ]:
values_for_alpha=[ 0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

(a). Estimer le vecteur de regression par la méthode Elastic-Net pour toutes les valeurs de alpha dans {0,0.1,0.2 ,...,1}

In [ ]:
elastic_net_cv=ElasticNetCV(alphas=values_for_alpha , l1_ratio=0.5 , cv=5 )

In [ ]:
elastic_net_cv.fit(x_train, y_train)

In [ ]:
best_alpha=elastic_net_cv.alpha_
print("Best alpha value:",best_alpha)

In [ ]:

y_pred_enet = elastic_net_cv.predict(x_test)


mse_enet = mean_squared_error(y_test, y_pred_enet)
r2_enet = r2_score(y_test, y_pred_enet)

print("Mean Squared Error (Elastic Net):", mse_enet)
print("R-squared (Elastic Net):", r2_enet)

In [ ]:



results = {}

for alpha in values_for_alpha:
    
    elastic_net = ElasticNet(alpha=alpha, l1_ratio=0.5)  
    elastic_net.fit(x_train, y_train)
    
    y_pred = elastic_net.predict(x_test)
    
    mse_elastic_net = mean_squared_error(y_test, y_pred)
    r2_elastic_net = r2_score(y_test, y_pred)
    
    results[alpha] = {
        "predictions": y_pred,
        "mse": mse_elastic_net,
        "r2": r2_elastic_net
    }

for alpha, metrics in results.items():
    print(f"Alpha: {alpha}")
    print(f"Mean Squared Error: {metrics['mse']}")
    print(f"R² Score: {metrics['r2']}")

(b).  Tracer le chemin de régularisation de LASSO

In [ ]:
alphas = np.logspace(-4, 1, 50)  # from 0.0001 to 10

In [ ]:
coefficients = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha)
    lasso.fit(x_train, y_train)
    
    # to store the coefficients
    coefficients.append(lasso.coef_)


coefficients = np.array(coefficients)


fig = go.Figure()


for i in range(coefficients.shape[1]):
    fig.add_trace(go.Scatter(
        x=alphas,
        y=coefficients[:, i],
        mode='lines',
        name=f'Feature {i+1}'
    ))
tickvals = [0.0001, 0.001, 0.01, 0.1, 1, 10]  
ticktext = [-4, -3, -2, -1, 0, 1]

fig.update_layout(
    height= 500,
    width= 500,
    title='Lasso Regularization Path',
    xaxis=dict(title='Alpha (Regularization Strength)', type='log',
                tickvals=tickvals , ticktext=ticktext , showline= True),
    yaxis=dict(title='Coefficient Value' , showline=True),
    showlegend= False,
    
    template='plotly_white'
)


fig.show()


(c): En utilisant seulement l'échantillon d'apprentissage , déterminer la valeur optimale de la paramètre de régularisation pour les trois méthods!

In [ ]:
# Pour LASSO
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42)

lasso_cv.fit(x_train, y_train)

optimal_lambda = lasso_cv.alpha_

print(f"Optimal Lambda for lasso : {optimal_lambda}")

In [ ]:
#Pour Ridge
mu_values=np.logspace(-4,1,10)
ridge_cv=RidgeCV(alphas=mu_values, cv=5)

ridge_cv.fit(x_train, y_train)
optimal_mu = ridge_cv.alpha_

print(f"Optimal Hyperparameter (mu) for Ridge: {optimal_mu}")

In [ ]:
alphas_enet = np.logspace(-4, 1, 10)  


l1_ratios = np.linspace(0, 1, 10)  

enet_cv = ElasticNetCV(alphas=alphas_enet, l1_ratio=l1_ratios, cv=5 , max_iter=10)


enet_cv.fit(x_train, y_train)


optimal_alpha = enet_cv.alpha_
optimal_l1_ratio = enet_cv.l1_ratio_

print(f"Optimal Alpha for Elastic Net: {optimal_alpha}")
print(f"Optimal L1 Ratio for Elastic Net: {optimal_l1_ratio}")

(d). La meilleure prédiction sur l'echantillon test????

In [ ]:
y_pred_lasso_cv=lasso_cv.predict(x_test)
y_pred_ridge_cv=ridge_cv.predict(x_test)
y_pred_enet_cv=enet_cv.predict(x_test)

In [ ]:
mse_lasso_cv= mean_squared_error(y_test, y_pred_lasso_cv)
mse_ridge_cv= mean_squared_error(y_test, y_pred_ridge_cv)
mse_enet_cv= mean_squared_error(y_test, y_pred_enet_cv)

r2_lasso_cv=r2_score(y_test, y_pred_lasso_cv)
r2_ridge_cv=r2_score(y_test, y_pred_ridge_cv)
r2_enet_cv=r2_score(y_test, y_pred_enet_cv)


print("MSE_lasso_cv:", mse_lasso_cv)
print("MSE_ridge_cv:", mse_ridge_cv)
print("MSE_enet_cv:", mse_enet_cv)

print("R2_score_lasso_cv:", r2_lasso_cv)
print("R2_score_ridge_cv:", r2_ridge_cv)
print("R2_score_enet_cv:", r2_enet_cv)


Question:2.   Memes question pour remplace beta par $\beta_1 =\beta_2=...=\beta_{1500}=1$ et les autres sont nulles.

In [ ]:
beta_2 =np.zeros(p)
beta_2[:1500]=1

X_2= np.random.randn(n,p)
Y_2=np.matmul(X , beta_2)+eta_vec

x2_train, x2_test, y2_train, y2_test=train_test_split(X_2,Y_2 , test_size=0.34, random_state=42)

In [ ]:
alphas_2=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
enet2=ElasticNetCV(alphas=alphas_2 , l1_ratio=0.5 , cv=5 )
enet2.fit(x2_train , y2_train)

In [ ]:
enet2_pred= enet2.predict(x2_test)

mse_enet2= mean_squared_error(y2_test , enet2_pred)
r2_enet2=r2_score(y2_test, enet2_pred)


print("Mean Squared Error (Elastic Net):", mse_enet2)
print("R-squared (Elastic Net):", r2_enet2)

In [ ]:
results = {}

for alpha in alphas_2:
    
    elastic_net = ElasticNet(alpha=alpha, l1_ratio=0.5)  
    
    
    elastic_net.fit(x2_train, y2_train)
    
   
    y2_pred = elastic_net.predict(x2_test)
    
   
    mse = mean_squared_error(y2_test, y2_pred)
    r2 = r2_score(y2_test, y2_pred)
    
    results[alpha] = {
        "predictions": y2_pred,
        "mse": mse,
        "r2": r2
    }


for alpha, metrics in results.items():
    print(f"Alpha: {alpha}")
    print(f"Mean Squared Error: {metrics['mse']}")
    print(f"R² Score: {metrics['r2']}")

In [ ]:
coefficients = []

alpha_values2 = np.logspace(-4, 1, 50)  

for alpha in alpha_values2:
    lasso = Lasso(alpha=alpha , max_iter=10000)
    lasso.fit(x2_train, y2_train)
    
    
    coefficients.append(lasso.coef_)
    
    



In [ ]:


coefficients = np.array(coefficients)


top_features = 2000
coefficients = coefficients[:, :top_features]


fig = go.Figure()

for i in range(coefficients.shape[1]):
    fig.add_trace(go.Scatter(
        x=alpha_values2,
        y=coefficients[:, i],
        mode='lines',
        name=f'Feature {i+1}' 
    ))


fig.update_layout(
    height=500,
    width=500,
    title='Lasso Regularization Path',
    xaxis=dict(
        title='Alpha (Regularization Strength)',
        type='log',
        tickvals=[0.0001, 0.001, 0.01, 0.1, 1, 10],
        ticktext=['0.0001', '0.001', '0.01', '0.1', '1', '10'],
        showline=True,
        range=[np.log10(0.0001), np.log10(10)]
    ),
    yaxis=dict(
        title='Coefficient Value',
        range=[-3, 3],
        showline=True
    ),
    showlegend=False,
    template='plotly_white'
)

# Display the figure
fig.show()



Question#3. 

In [ ]:
n=100
p=50

beta3=np.zeros(p)
beta3[:14]=1


In [ ]:
cov_matrix=np.zeros((p,p))

for i in range(p):
    for j in range(p):
        cov_matrix[i, j] = 0.7 ** abs(i - j)

In [ ]:
mean = np.zeros(p)  
X3 = np.random.multivariate_normal(mean, cov_matrix, size=n)
eta_vec3=np.random.normal(0,1,n)
Y3= np.matmul(X3,beta3)+eta_vec3
x3_train,x3_test, y3_train, y3_test = train_test_split(X3,Y3 , test_size=0.34 , random_state=42)

In [ ]:
alphas_3=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
enet3=ElasticNetCV(alphas=alphas_3 , l1_ratio=0.5 , cv=5 )
enet3.fit(x3_train , y3_train)

In [ ]:
enet3_pred= enet3.predict(x3_test)

mse_enet3= mean_squared_error(y3_test , enet3_pred)
r2_enet3=r2_score(y3_test, enet3_pred)


print("Mean Squared Error (Elastic Net):", mse_enet3)
print("R-squared (Elastic Net):", r2_enet3)

In [ ]:
results = {}

for alpha in alphas_3:
   
    elastic_net = ElasticNet(alpha=alpha, l1_ratio=0.5) 
    
   
    elastic_net.fit(x3_train, y3_train)
    
 
    y3_pred = elastic_net.predict(x3_test)
    
   
    mse = mean_squared_error(y3_test, y3_pred)
    r2 = r2_score(y3_test, y3_pred)
    
    results[alpha] = {
        "predictions": y3_pred,
        "mse": mse,
        "r2": r2
    }


for alpha, metrics in results.items():
    print(f"Alpha: {alpha}")
    print(f"Mean Squared Error: {metrics['mse']}")
    print(f"R² Score: {metrics['r2']}")
    


In [ ]:
coefficients = []

alpha_values3 = np.logspace(-4, 1, 40)  
for alpha in alpha_values3:
  
    lasso = Lasso(alpha=alpha , max_iter=10000)
    lasso.fit(x3_train, y3_train)
    
    
    coefficients.append(lasso.coef_)

coefficients = np.array(coefficients)


top_features = 5000
coefficients = coefficients[:, :top_features]


fig = go.Figure()


for i in range(coefficients.shape[1]):
    fig.add_trace(go.Scatter(
        x=alpha_values2,
        y=coefficients[:, i],
        mode='lines',
        name=f'Feature {i+1}' 
    ))


fig.update_layout(
    height=800,
    width=800,
    title='Lasso Regularization Path',
    xaxis=dict(
        title='Alpha (Regularization Strength)',
        type='log',
        tickvals=[0.0001, 0.001, 0.01, 0.1, 1, 10],
        ticktext=['0.0001', '0.001', '0.01', '0.1', '1', '10'],
        showline=True,
        range=[np.log10(0.0001), np.log10(10)]
    ),
    yaxis=dict(
        title='Coefficient Value',
        range=[-1, 2],
        showline=True
    ),
    showlegend=False,
    template='plotly_white'
)

# Display the figure
fig.show()

In [ ]:
df = pd.read_csv("garments_worker_productivity.csv")

df=df.drop(columns=['date', 'quarter', 'department' , 'day'])

X= df.drop(columns=['actual_productivity'])
y= df['actual_productivity']

X_filled= X.fillna(0)
y_filled = y.fillna(0)


x_train_df,x_test_df , y_train_df, y_test_df=train_test_split(X_filled,y_filled , test_size=0.34 , random_state=42)
print("Training_sample shape:", x_train_df.shape)
print("test_sample shape:", x_test_df.shape)
print("Training_labels shape:", y_train_df.shape)
print("test_labels shape:", y_test_df.shape)

part (a).

In [ ]:
alpha_values_df=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

In [ ]:
elastic_net_cv_df=ElasticNetCV(alphas=alpha_values_df , l1_ratio=0.5 , cv=5 )

In [ ]:
elastic_net_cv_df.fit( x_train , y_train )

In [ ]:
best_alpha_df=elastic_net_cv_df.alpha_
print("Best alpha value:",best_alpha_df)

In [ ]:
results = {}

for alpha in alpha_values_df:
    
    elastic_net = ElasticNet(alpha=alpha, l1_ratio=0.5) 
    
   
    elastic_net.fit(x_train_df, y_train_df)
    
    
    y_pred_df = elastic_net.predict(x_test_df)
    
    
    mse = mean_squared_error(y_test_df, y_pred_df)
    r2 = r2_score(y_test, y_pred)
    
    results[alpha] = {
        "predictions": y_pred_df,
        "mse": mse,
        "r2": r2
    }


for alpha, metrics in results.items():
    print(f"Alpha: {alpha}")
    print(f"Mean Squared Error: {metrics['mse']}")
    print(f"R² Score: {metrics['r2']}")

part (b)

In [ ]:

coefficients = []

for alpha in alphas:
    
    lasso = Lasso(alpha=alpha)
    lasso.fit(x_train_df, y_train_df)
    
   
    coefficients.append(lasso.coef_)


coefficients = np.array(coefficients)


fig = go.Figure()


for i in range(coefficients.shape[1]):
    fig.add_trace(go.Scatter(
        x=alphas,
        y=coefficients[:, i],
        mode='lines',
        name=f'Feature {i+1}'
    ))
tickvals = [0.0001, 0.001, 0.01, 0.1, 1, 10]  
ticktext = [-4, -3, -2, -1, 0, 1]

fig.update_layout(
    height= 800,
    width= 800,
    title='Lasso Regularization Path',
    xaxis=dict(title='Alpha (Regularization Strength)', type='log',
                tickvals=tickvals , ticktext=ticktext , showline= True),
    yaxis=dict(title='Coefficient Value' , showline=True),
    showlegend= False,
  
    template='plotly_white'
)


fig.show()

part (c).

In [ ]:
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42)

lasso_cv.fit(x_train_df, y_train_df)

optimal_lambda_df = lasso_cv.alpha_

print(f"Optimal Lambda for lasso for real data : {optimal_lambda_df}")

In [ ]:
#Pour Ridge
mu_values=np.logspace(-4,1,10)
ridge_cv=RidgeCV(alphas=mu_values, cv=5)

ridge_cv.fit(x_train_df, y_train_df)
optimal_mu_df = ridge_cv.alpha_

print(f"Optimal Hyperparameter (mu) for Ridge for real data : {optimal_mu_df}")

In [ ]:
alphas_enet = np.logspace(-4, 1, 50)  


l1_ratios = np.linspace(0, 1, 10)  

enet_cv = ElasticNetCV(alphas=alphas_enet, l1_ratio=l1_ratios, cv=5 , max_iter=10)


enet_cv.fit(x_train_df, y_train_df)


optimal_alpha = enet_cv.alpha_
optimal_l1_ratio = enet_cv.l1_ratio_

print(f"Optimal Alpha for Elastic Net for real data : {optimal_alpha}")
print(f"Optimal L1 Ratio for Elastic Net for real data : {optimal_l1_ratio}")

part (d).

In [ ]:
y_pred_lasso_cv=lasso_cv.predict(x_test_df)
y_pred_ridge_cv=ridge_cv.predict(x_test_df)
y_pred_enet_cv=enet_cv.predict(x_test_df)

In [ ]:
mse_lasso_cv= mean_squared_error(y_test_df, y_pred_lasso_cv)
mse_ridge_cv= mean_squared_error(y_test_df, y_pred_ridge_cv)
mse_enet_cv= mean_squared_error(y_test_df, y_pred_enet_cv)

r2_lasso_cv=r2_score(y_test_df, y_pred_lasso_cv)
r2_ridge_cv=r2_score(y_test_df, y_pred_ridge_cv)
r2_enet_cv=r2_score(y_test_df, y_pred_enet_cv)


print("MSE_lasso_cv:", mse_lasso_cv)
print("MSE_ridge_cv:", mse_ridge_cv)
print("MSE_enet_cv:", mse_enet_cv)

print("R2_score_lasso_cv:", r2_lasso_cv)
print("R2_score_ridge_cv:", r2_ridge_cv)
print("R2_score_enet_cv:", r2_enet_cv)